# Fine-tune a Model and Evaluate it using ROUGE Metrics

* We mark **TODO** in the notebook cells to indicate the place where you need to complete the missing code. You can refer to the exercises in the course repository for code examples.

## Install necessary packages

This is a onestep process to install necessary bitsandbytes(Alpha release) for the notebook.
1. Run below cell uncommenting the installation commands, after successful installation, comment back again.
2. Now Restart the kernel. `Kernel->Restart Kernel`
3. Now run the cells normally.

In [2]:
import sys
import os
import site
from pathlib import Path

!echo "Installation in progress, please wait..."
!{sys.executable} -m pip cache purge > /dev/null

%pip install --user --upgrade transformers datasets trl peft accelerate scipy sentencepiece ipywidgets evaluate rouge_score --no-warn-script-location

!echo "Installation completed."

# Get the site-packages directory
site_packages_dir = site.getsitepackages()[0]

# add the site pkg directory where these pkgs are insalled to the top of sys.path
if not os.access(site_packages_dir, os.W_OK):
    user_site_packages_dir = site.getusersitepackages()
    if user_site_packages_dir in sys.path:
        sys.path.remove(user_site_packages_dir)
    sys.path.insert(0, user_site_packages_dir)
else:
    if site_packages_dir in sys.path:
        sys.path.remove(site_packages_dir)
    sys.path.insert(0, site_packages_dir)

Installation in progress, please wait...
Note: you may need to restart the kernel to use updated packages.
Installation completed.


## Import necessary packages

In [3]:
import torch
import os

os.environ["WANDB_DISABLED"] = "true"
import transformers
from transformers import AutoTokenizer
from peft import LoraConfig
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
from peft import get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from evaluate import load

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


## Login to HuggingFace

In [4]:
from huggingface_hub import notebook_login
notebook_login()

## Load Gemma-2-2b-it Model from HuggingFace Hub

In [5]:
model_path = "google/gemma-2-2b-it"

# TODO: create tokenizer using AutoTokenizer class
# tokenizer = ...
tokenizer = AutoTokenizer.from_pretrained(model_path, device_map="auto", use_fast=True)

model = AutoModelForCausalLM.from_pretrained(model_path,
                                             attn_implementation='eager',
                                             device_map="auto")


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Load, Format, and Split Dataset

In [6]:
def process_dataset(sample):
    messages = [
        {"role": "user", "content": f"Instruction:\nSummarize the following article.\n\nInput:\n{sample['Articles']}"},
        {"role": "assistant", "content": sample['Summaries']}
    ]
    sample = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True)
    return sample

dataset = load_dataset("gopalkalpande/bbc-news-summary", split="train")
dataset = dataset.map(process_dataset)

split_dataset = dataset.train_test_split(test_size=0.1, seed=99)
train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]

## Evaluate Base Model Summaries using ROUGE Metric

In [7]:
rouge = load('rouge')

# initialize lists of predictions and references later used to compute rouge scores
predictions = []
references = []

# iterate through the first 15 samples
for article, abstract in zip(validation_dataset["Articles"][:15], validation_dataset["Summaries"][:15]):
    messages = [
        {"role": "user", "content": f"Instruction:\nSummarize the following article.\n\nInput:\n{article}"},
    ]
    input_ids = tokenizer.apply_chat_template(messages,
                                              tokenize=True,
                                              add_generation_prompt=True,
                                              return_tensors="pt").to("cuda")
    
    # TODO: perform model inference using the tokens in ``input_ids''
    # output =   
    output = model.generate(input_ids=input_ids, max_new_tokens=128, do_sample=False)   
    
    # Remove input prompt from output
    prompt_length = input_ids.shape[1]
    answer = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)
    
    # TODO: add one answer to the ``predictions'' list, which is later passed to rouge compute
    predictions.append(answer)
    
    # TODO: add one abstract to the ``references'' list, which is later passed to rouge compute
    references.append(abstract)
    
    print(100*'-')
    print("Abstract:", abstract)
    print(100*'-')
    print("Model Summary:", answer)

print(100*'-')
# TODO: compute and print out the rouge scores including rouge1, rouge2, rougeL and rougeLsum
# TODO: you can refer to https://huggingface.co/spaces/evaluate-metric/rouge/blob/main/README.md#how-to-use
# print(...)

scores = rouge.compute(
    predictions=predictions,
    references=references,
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
    use_stemmer=True,
)
print({k: round(v, 4) for k, v in scores.items()})

print(100*'-')



----------------------------------------------------------------------------------------------------
Abstract: Immigration and asylum have normally been issues politicians from the big parties have tiptoed around at election time.But, while all the parties appear to agree the time has come to properly debate and address the issue, there are already signs they will run into precisely the same problems as before.Labour has already branded the proposal unworkable but party strategists have seen the Tories seizing a poll advantage over the issue.Former union leader Sir Bill Morris has already accused both the big parties of engaging in a "bidding war about who can be nastiest to asylum seekers".The challenge for the big parties is to ensure they can engage in the debate during the cut and thrust of a general election while also avoiding that trap.That has been attacked by the Tories as too little, too late and for failing to tackle the key issue of the numbers entering the UK.That was also

## Run the SFTTrainer to Fine-tune Model

In [10]:
finetuned_model = "gemma-2-2b-it-finetuned"

peft_config = LoraConfig(
    r=64,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    modules_to_save=["lm_head", "embed_token"],
    task_type="CAUSAL_LM",
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# TODO: set up the trainer using SFTTrainer class
# TODO: you can refer to the gemma_xpu_finetuning.ipynb exercise
# TODO: this part is relatively long because of the arguments that need to be set
# trainer = SFTTrainer(...)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=SFTConfig(
    output_dir=finetuned_model,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=20,
    eval_strategy="no", 
    max_length=128, 
    packing=True,
    report_to="none",
    dataset_text_field=None,
    ),
)


model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
result = trainer.train()
model.config.use_cache = True
print(result)

/home/exouser/.local/lib/python3.12/site-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/home/exouser/.local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/exouser/.local/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


Packing train dataset:   0%|          | 0/2001 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/223 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.20 GiB. GPU 0 has a total capacity of 20.00 GiB of which 2.00 GiB is free. Including non-PyTorch memory, this process has 16.34 GiB memory in use. Of the allocated memory 15.18 GiB is allocated by PyTorch, and 667.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Inference Fine-tuned Model and Evaluate Summaries using ROUGE

In [ ]:
rouge = load('rouge')

finetuned_model_path = f"{finetuned_model}/checkpoint-500"
loaded_model = AutoModelForCausalLM.from_pretrained(finetuned_model_path, device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path)

predictions = []
references = []

# TODO: compute rouge scores on the first 15 sample again.
# TODO: you can repeat the code from the earlier cells.
#

for article, abstract in zip(validation_dataset["Articles"][:15], validation_dataset["Summaries"][:15]):
    messages = [
        {"role": "user", "content": f"Instruction:\nSummarize the following article.\n\nInput:\n{article}"},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    output = model.generate(input_ids=input_ids, max_new_tokens=128, do_sample=False)
    prompt_length = input_ids.shape[1]
    answer = tokenizer.decode(output[0][prompt_length:], skip_special_tokens=True)
    predictions.append(answer)
    references.append(abstract)

scores = rouge.compute(
    predictions=predictions,
    references=references,
    rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"],
    use_stemmer=True,
)
print({k: round(v, 4) for k, v in scores.items()})